# ROSP-9: System availability / uptime

PTH Tier 1 metric **ROSP-9** - {{ params.start_dayobs }} to {{ params.end_dayobs }} (dayObs, inclusive)

- Jira epic: [ROSP-9](https://rubinobs.atlassian.net/browse/ROSP-9)
- Confluence (Hub page; no metric page yet): [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

| | |
|---|---|
| Steward | Erik Dennihy |
| Owner | Sandrine Thomas |
| Category | 1 - Observatory Operations |
| Layer | 2 |
| Target | 90% availability of observable nights; &le;14 days scheduled and &le;21 days unscheduled downtime per year |

Method version: v1.0 (draft)

Status: first implementation, **partial**. The availability fraction is computed, and planned and unplanned downtime are reported separately for the nights covered by the shared observatory state overrides (a partial resolution of ROSP-28). The subsystem attribution is not yet possible (see [Results](#results)). Changes to the method go through the PTH Method Change Process: https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501

## What this notebook measures

> Of the observable night time that was **not lost to weather**, what fraction of the time was the observatory **not in `FAULT` or `DOWNTIME`**?

$$\text{availability} = \frac{\text{observable night} - \text{weather} - (\text{fault or downtime outside weather})}{\text{observable night} - \text{weather}}$$

**Read this number together with time-loss accounting ([ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11)).** An availability figure on its own does not say *why* time was lost. Weather is removed from the denominator here precisely so that weather is not charged against the engineering systems.

## How the notebook is organised

The sections follow the same order as the question above:

1. **Data sources:** where the night boundaries, the observatory states and the state overrides come from.
2. **Selection:** which time counts. Start from the observable night, remove weather; what remains is the denominator.
3. **Computation:** inside that selected time, how much was the system available, and how much of the downtime was planned?
4. **Results:** the headline number against the 90% target, and a calendar showing it night by night.
5. **Configuration-change markers:** notes on changes to the system or the method that could explain steps in the series.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
# Defaults are the PTH baseline epoch.
start_dayobs = 20260629  # First dayObs (inclusive)
end_dayobs = 20261231  # Last dayObs (inclusive)

not_times_square = True

In [ ]:
# Validate the dayObs range and report it. Times Square substitutes the
# parameters above as Python ints, but coerce anyway so a string survives.
from datetime import datetime

start_dayobs = int(start_dayobs)
end_dayobs = int(end_dayobs)

if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )

start_date = datetime.strptime(str(start_dayobs), "%Y%m%d").date()
end_date = datetime.strptime(str(end_dayobs), "%Y%m%d").date()
n_nights = (end_date - start_date).days + 1

print(f"dayObs range: {start_dayobs} to {end_dayobs} (inclusive)")
print(f"Dates: {start_date.isoformat()} to {end_date.isoformat()} ({n_nights} nights)")

In [ ]:
# Times Square runs this notebook in a Nublado pod where rubin_nights
# may be missing or stale; install/upgrade it there. When running
# anywhere else (e.g. an SDF login node) this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

## Data sources

Three inputs are needed, and they answer different questions.

| Input | Where it comes from |
|---|---|
| **Twilight Boundary** | Computed with `rubin_nights.dayobs_utils` (no database needed) |
| **Observatory State** | The EFD `observatoryStatus` topic, via `rubin_nights.observatory_status` |
| **Observatory state overrides** | `tracking_hub/ops/observatory_state_overrides/observatory_state_overrides.csv`, shared with other notebooks |

### 1. The observable night

For each dayObs, the observable night runs from the moment the Sun is at altitude **-12 deg** in the evening to the moment it is at **-12 deg** in the morning. This is the definition chosen in [ROSP-22](https://rubinobs.atlassian.net/browse/ROSP-22) (notebook `observable_night_definition_ROSP-22`). It depends only on the Sun, so it is the same for every year and does not change with weather, faults or the dome. The definition must be held fixed: changing it later is a series break, not a restatement.

### 2. The observatory state

The observatory state is recorded in the EFD. The possible states are:

| State | Meaning for this notebook |
|---|---|
| `WEATHER` | Time lost to weather. **Removed from the denominator** (Selection). |
| `OPERATIONAL` | The observatory is available and observing. **Available.** |
| `IDLE` | The observatory is available but not observing. **Available.** |
| `UNKNOWN` | The state could not be determined. **Treated as available** (it is not recorded as a fault). |
| `FAULT` | A fault is active. **Not available, unplanned.** |
| `DOWNTIME` | The observatory is in declared downtime. **Not available, unplanned** (unless an override marks it planned, below). |
| `PLANNED_DOWNTIME` | Not an EFD state. `DOWNTIME` that an override marks as planned. **Not available, planned.** |

### 3. The observatory state overrides

The EFD `DOWNTIME` state is a mix of scheduled (planned) and intentional but unplanned observatory shutdown, and for some periods the recorded state is simply wrong or missing. The shared overrides file replaces the EFD state of whole nights (12:00 UT to 12:00 UT, which covers the whole -12 to -12 deg night) and flags `DOWNTIME` that was **planned**. That gives a first separation of planned and unplanned downtime, a partial resolution of [ROSP-28](https://rubinobs.atlassian.net/browse/ROSP-28). It only exists for the nights in the file; elsewhere the EFD `DOWNTIME` cannot be classified and counts as unplanned.

The next cells load the tools, define the observable night, query the states, and apply the overrides.

In [ ]:
# Imports, and the connection to the EFD.
# Imports live here (not in the parameters cell) because Times Square replaces the parameters cell wholesale.
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.time import Time, TimeDelta

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
import rubin_nights.observatory_status as observatory_status

# The Sun-position code needs the shared rubin_sim data. If it is not set, point at the shared copy at USDF.
if os.getenv("RUBIN_SIM_DATA_DIR") is None:
    for _data_dir in (
        "/sdf/data/rubin/shared/rubin_sim_data",
        "/sdf/group/rubin/shared/rubin_sim_data",
    ):
        if os.path.isdir(_data_dir):
            os.environ["RUBIN_SIM_DATA_DIR"] = _data_dir
            break

# On an RSP (including Times Square) the default endpoints and tokens are right; elsewhere get_clients() looks for
# your own USDF token (see rubin_nights.get_access_token).
if os.getenv("EXTERNAL_INSTANCE_URL", "") != "":
    tokenfile = None
    site = None
endpoints = connections.get_clients()

In [ ]:
# THE DEFINITION OF THE OBSERVABLE NIGHT (from ROSP-22): Sun at altitude -12 deg in the evening to -12 deg in the morning.
TWILIGHT_ALT = (
    -12.0
)  # Sun altitude (deg) at the start (evening) and end (morning) of the observable night


def observable_night(day_obs):
    """Start and end of the observable night of one dayObs, as astropy Times."""
    start, end = rn_dayobs.day_obs_sunset_sunrise(day_obs, TWILIGHT_ALT)
    return start, end

In [ ]:
# The list of nights to analyse: every dayObs from start_dayobs to end_dayobs, but not nights that have not happened yet.
today_dayobs = rn_dayobs.day_obs_str_to_int(rn_dayobs.today_day_obs())
last_dayobs = min(
    end_dayobs, today_dayobs
)  # a night in the future has no states to look at


def day_obs_range(first_day_obs, last_day_obs):
    """All dayObs (as YYYYMMDD integers) from first_day_obs to last_day_obs, inclusive."""
    days = pd.date_range(
        pd.to_datetime(str(first_day_obs), format="%Y%m%d"),
        pd.to_datetime(str(last_day_obs), format="%Y%m%d"),
    )
    return [int(d.strftime("%Y%m%d")) for d in days]


day_obs_list = day_obs_range(start_dayobs, last_dayobs)

# Times that bracket all of those nights, for the EFD query (dayObs starts at 12:00 UTC of its date).
t_start = rn_dayobs.day_obs_to_time(start_dayobs)
t_end = rn_dayobs.day_obs_to_time(last_dayobs) + TimeDelta(1, format="jd")
print(f"{len(day_obs_list)} nights, dayObs {start_dayobs} to {last_dayobs}")
if last_dayobs < end_dayobs:
    print(
        f"(end_dayobs {end_dayobs} is in the future; the analysis stops at {last_dayobs})"
    )

### Querying the observatory states

`get_observatory_state_times` returns the EFD states as **periods**: one row per stretch of time (`start`, `end`) during which a state (`type`) was in force, together with the night it belongs to (`day_obs`) and that night's twilight times (`sunset12`, `sunrise12`).

**These periods overlap.** A `WEATHER` period typically spans the whole night while `OPERATIONAL`, `IDLE` or `FAULT` periods run inside it, so the hours of the states do *not* add up to the length of the night. We therefore never add per-state hours together. Instead, later cells take the **union** of the periods of interest (the set of moments covered by at least one of them), which counts every moment once.

In [ ]:
# Query the EFD for the state periods of all nights in the range.
obs_status_periods = observatory_status.get_observatory_state_times(
    t_start, t_end, endpoints["efd"]
)
print(
    f"{len(obs_status_periods)} state periods on {obs_status_periods.day_obs.nunique()} of {len(day_obs_list)} nights"
)
display(obs_status_periods.head(10))

### Correcting the raw state record: the overrides

The EFD logs `observatoryStatus` only when the state *changes*, and the library builds periods only for nights that have a message. A quiet night therefore vanishes from the statistics, which would make the availability look better than it was. ROSP-22 handled this by carrying the last known state forward over quiet nights. **This notebook does not carry anything forward**, because that is an assumption (a long gap could also be a hole in the EFD record). Instead:

1. **Observatory state overrides replace the logged state of whole nights.** The overrides live in a shared file, `tracking_hub/ops/observatory_state_overrides/observatory_state_overrides.csv` (see the README in that folder), so that other notebooks can apply the same ones. There is one row per UT date, with the window it applies to (12:00 UT on that date to 12:00 UT on the next, which covers the whole -12 to -12 deg night), the states in force, and whether `DOWNTIME` is **planned**. **Where an override exists it always replaces what the EFD gave for that night**, and it creates the night if the EFD has no messages for it. Planned downtime is given its own label, `PLANNED_DOWNTIME`, so that it can be reported separately from unplanned `DOWNTIME` (a partial resolution of [ROSP-28](https://rubinobs.atlassian.net/browse/ROSP-28)).

   The overrides currently cover 15 July to 25 October 2026:

   | dayObs | State |
   |---|---|
   | 20260715 to 20260831 | `DOWNTIME \| WEATHER` |
   | 20260901 to 20260914 | `WEATHER` or `DOWNTIME`, per the ROSP-53 aggregated weather loss file |
   | 20260915 to 20261004 | planned `DOWNTIME` |
   | 20261005 to 20261025 | `WEATHER` or `DOWNTIME`, per the ROSP-53 aggregated weather loss file |

   After 25 October the state was recorded correctly on the summit and no override is needed. As before, weather is removed first, so a night in `WEATHER` drops out of the selection and its downtime is not charged to the system (see Checks on the computation).
2. **A warning, not a fill, for any night still without a state.** After the overrides, any night in the range that has no EFD messages and no override is listed in a warning and is **left out of the statistics**. If the state of such a night is known, add it to the CSV.

When this was written, every one of the 77 quiet nights in 29 June to 28 October 2026 was covered by an override, so nothing triggers the warning.

In [ ]:
# The observatory state overrides: a shared CSV with one row per overridden night (see its README).
# Notebooks here must not import from each other, but they can all read the same data file.
import warnings
from pathlib import Path

OVERRIDES_CSV = Path("../observatory_state_overrides/observatory_state_overrides.csv")
PLANNED_DOWNTIME = "PLANNED_DOWNTIME"  # state label used below for DOWNTIME rows flagged planned_downtime = TRUE


def read_state_overrides(path):
    """Read the override CSV: one row per night, with the states to apply and whether DOWNTIME is planned."""
    ov = pd.read_csv(
        path,
        dtype={"day_obs": int, "states": str, "planned_downtime": str},
        parse_dates=["start_utc", "end_utc"],
        keep_default_na=False,
    )
    ov["planned_downtime"] = ov.planned_downtime.str.upper() == "TRUE"
    # The override must cover the whole dayObs (12:00 UT to 12:00 UT), i.e. the whole -12 to -12 deg night.
    expected_start = pd.to_datetime(
        ov.day_obs.astype(str), format="%Y%m%d"
    ) + pd.Timedelta(hours=12)
    bad = ov[
        (ov.start_utc != expected_start)
        | (ov.end_utc != expected_start + pd.Timedelta(days=1))
    ]
    if len(bad):
        raise ValueError(
            f"override windows must be 12:00 UT to 12:00 UT of dayObs: {bad.day_obs.tolist()}"
        )
    if ov.day_obs.duplicated().any():
        raise ValueError(
            f"duplicate override nights: {ov.day_obs[ov.day_obs.duplicated()].tolist()}"
        )
    return ov


def override_types(row):
    """The state types of one override row. Planned DOWNTIME gets its own label so it can be reported separately."""
    types = row.states.split("|")
    if row.planned_downtime:
        types = [PLANNED_DOWNTIME if t == "DOWNTIME" else t for t in types]
    return types


def apply_state_overrides(periods, overrides, nights_in_range):
    """Replace all EFD periods of each overridden night by one full-night period per override state.

    A night with no EFD periods (a quiet night) is created from the Sun definition. A night that has not finished is
    clipped at the current time and a night that has not started is skipped.
    """
    tnow = Time.now().utc
    have = periods.groupby("day_obs")[["sunset12", "sunrise12"]].first()
    rows, replaced, created = [], [], []
    for row in overrides[overrides.day_obs.isin(nights_in_range)].itertuples():
        d = row.day_obs
        if d in have.index:
            sunset12, sunrise12 = have.loc[d, "sunset12"], have.loc[d, "sunrise12"]
            replaced.append(d)
        else:
            sunset, sunrise = (
                t.utc for t in rn_dayobs.day_obs_sunset_sunrise(d, TWILIGHT_ALT)
            )
            if tnow <= sunset:
                continue  # night has not started yet
            sunset12, sunrise12 = sunset.datetime, min(sunrise, tnow).datetime
            created.append(d)
        hours = (sunrise12 - sunset12).total_seconds() / 3600
        for t in override_types(row):
            rows.append(
                {
                    "day_obs": d,
                    "sunset12": sunset12,
                    "sunrise12": sunrise12,
                    "start": sunset12,
                    "end": sunrise12,
                    "hours": hours,
                    "type": t,
                }
            )
    done = set(replaced) | set(created)
    periods = pd.concat(
        [periods[~periods.day_obs.isin(done)], pd.DataFrame(rows)], ignore_index=True
    )
    return (
        periods.sort_values(["day_obs", "start"], ignore_index=True),
        replaced,
        created,
    )


def nights_without_state(periods, nights_in_range):
    """Nights in the range that have no state periods at all (and have already started)."""
    tnow = Time.now().utc
    missing = sorted(set(nights_in_range) - set(periods.day_obs.unique()))
    return [
        d
        for d in missing
        if rn_dayobs.day_obs_sunset_sunrise(d, TWILIGHT_ALT)[0].utc < tnow
    ]


state_overrides = read_state_overrides(OVERRIDES_CSV)
print(
    f"{len(state_overrides)} override nights in {OVERRIDES_CSV.name}: "
    f"{state_overrides.day_obs.min()} to {state_overrides.day_obs.max()}"
)

# Where an override exists it replaces whatever the EFD gave for that night, and creates the night if the EFD has no
# status messages for it. There is no carry-forward: a night with no EFD messages and no override has no state.
efd_quiet_nights = sorted(set(day_obs_list) - set(obs_status_periods.day_obs.unique()))
obs_status_periods, override_replaced, override_created = apply_state_overrides(
    obs_status_periods, state_overrides, day_obs_list
)
print(
    f"overrides applied to {len(override_replaced) + len(override_created)} nights: "
    f"{len(override_replaced)} replaced EFD states, {len(override_created)} on nights with no EFD status messages"
)
print(
    f"{len(efd_quiet_nights)} of {len(day_obs_list)} nights have no EFD status messages"
)

# The EFD logs observatoryStatus only when the state changes, so a quiet night is not an error in itself, but this
# notebook no longer assumes the previous state persisted. Any night still without a state drops out of the statistics.
no_state_nights = nights_without_state(obs_status_periods, day_obs_list)
if no_state_nights:
    warnings.warn(
        f"{len(no_state_nights)} nights have no observatory state (no EFD status messages and no override) and are "
        f"NOT in the statistics: {no_state_nights}. Add them to observatory_state_overrides.csv if the state is known.",
        stacklevel=2,
    )
else:
    print("every night in the range has a state")

## Selection

Selection decides **which time counts** towards the availability. It is the denominator, and it determines the answer at least as much as the numerator does, so it is done in two explicit steps:

1. **Start from the observable night.** For every dayObs, the time between the Sun reaching -12 deg in the evening and -12 deg in the morning (Data sources, above).
2. **Remove the time lost to weather.** Time in the `WEATHER` state is taken out. Weather is outside the control of the observatory's systems, and leaving it in would charge weather losses against engineering.

What is left, *observable night minus weather*, is the **selected time**. The availability is measured only within it.

> Weather is removed here, for this metric, because the question is about the *system*. This does not change the definition of the observable night itself (ROSP-22), which is the Sun alone.

In [ ]:
# STEP 1: the observable night of every night in the range.
# The state periods carry each night's -12 deg twilight times (sunset12, sunrise12), which are the same Sun-only
# definition as above. We take them from there so the night window and the state periods use identical timestamps.
nights = obs_status_periods.groupby("day_obs")[["sunset12", "sunrise12"]].first()
nights = nights.rename(columns={"sunset12": "night_start", "sunrise12": "night_end"})
nights["night_hours"] = (
    nights.night_end - nights.night_start
).dt.total_seconds() / 3600

# Cross-check against the function that defines the observable night, on the nights that have completely finished
# (a night created by an override while it is still in progress is clipped at the current time, so it is shorter).
complete = nights[nights.night_end < pd.Timestamp.utcnow().tz_localize(None)]
sun_hours = pd.Series(
    {
        d: (lambda s_e: (s_e[1] - s_e[0]).jd * 24)(observable_night(d))
        for d in complete.index
    }
)
print(
    f"{len(nights)} nights with state data; longest difference between the state-table night and the "
    f"Sun definition: {(complete.night_hours - sun_hours).abs().max() * 60:.2f} minutes"
)
display(nights.round(2).head())

In [ ]:
# A helper used from here on: how many hours of a night are covered by periods of the given state types?
# Periods can overlap, so this takes the UNION: it walks the periods in time order and counts each moment once.
def union_hours(periods, types, lo, hi):
    """Hours covered by the union of `types` periods, clipped to the window [lo, hi]."""
    d = periods[periods.type.isin(types)].sort_values("start")
    total, cur_end = 0.0, lo
    for s, e in zip(d.start, d.end):
        s, e = max(s, lo, cur_end), min(e, hi)
        if e > s:
            total += (e - s).total_seconds() / 3600
            cur_end = e
    return total


# Group the periods by night once, so each night's periods are easy to pick out.
periods_by_night = {d: g for d, g in obs_status_periods.groupby("day_obs")}

In [ ]:
# STEP 2: remove the time lost to weather. selected time = observable night - weather.
nights["weather_hours"] = [
    union_hours(periods_by_night[d], ["WEATHER"], w.night_start, w.night_end)
    for d, w in nights.iterrows()
]
nights["selected_hours"] = nights.night_hours - nights.weather_hours

display(nights[["night_hours", "weather_hours", "selected_hours"]].round(2).head(10))

In [ ]:
# What the selection kept, over the whole range.
total_night = nights.night_hours.sum()
total_weather = nights.weather_hours.sum()
total_selected = nights.selected_hours.sum()
n_weathered_out = int((nights.selected_hours < 0.05).sum())

print(f"Observable night:       {total_night:8.1f} h over {len(nights)} nights")
print(
    f"Lost to weather:        {total_weather:8.1f} h ({total_weather / total_night:.1%} of the observable night)"
)
print(
    f"Selected (night - wx):  {total_selected:8.1f} h ({total_selected / total_night:.1%} of the observable night)"
)
print(f"Nights with no selected time (entirely weather): {n_weathered_out}")

## Computation

Now we ask, **inside the selected time**, how much of it the system was available.

The rule is the one from the Data sources table:

- **Available:** the observatory is in `OPERATIONAL`, `IDLE` or `UNKNOWN`.
- **Not available, unplanned:** the observatory is in `FAULT` or `DOWNTIME`.
- **Not available, planned:** the observatory is in `PLANNED_DOWNTIME` (planned `DOWNTIME` from the overrides, see [ROSP-28](https://rubinobs.atlassian.net/browse/ROSP-28)).

Planned and unplanned time both count as not available in the availability fraction, as before. They are reported separately so they can be compared with their own yearly allocations.

Because the periods overlap, we count the *not available* time and subtract it, instead of adding up the available states:

$$\text{available} = \text{selected} - \text{not available},\qquad \text{availability} = \frac{\text{available}}{\text{selected}}$$

**How to count "not available outside weather".** We need the hours that are in `FAULT` or `DOWNTIME` (planned or not) *and not* in `WEATHER`. With union lengths $U(\cdot)$, this is

$$\bigl|(F \cup D \cup P) \setminus W\bigr| \;=\; U(F \cup D \cup P \cup W) - U(W)$$

that is, add the fault and downtime periods to the weather periods, and subtract the weather alone. Whatever is left was fault or downtime that happened while the weather was fine. Fault or downtime that overlaps weather is not charged to the system.

The planned part is $U(P \cup W) - U(W)$, and the unplanned part is the rest. Where a planned and an unplanned period overlap, the time is counted as planned, so the two parts always add up to the total.

The cells below do this one step at a time, per night, then pool the nights into the final fraction.

In [ ]:
# The states that count as available and as not available. Change them here and nothing else needs to change.
# Not available is split in two: unplanned (FAULT, unplanned DOWNTIME) and planned (PLANNED_DOWNTIME, from the overrides).
AVAILABLE_STATES = ["OPERATIONAL", "IDLE", "UNKNOWN"]
UNPLANNED_STATES = ["FAULT", "DOWNTIME"]
PLANNED_STATES = [PLANNED_DOWNTIME]
NOT_AVAILABLE_STATES = UNPLANNED_STATES + PLANNED_STATES
WEATHER_STATES = ["WEATHER"]

print(f"available:                {AVAILABLE_STATES}")
print(f"not available, unplanned: {UNPLANNED_STATES}")
print(f"not available, planned:   {PLANNED_STATES}")
print(f"removed first:            {WEATHER_STATES}")

In [ ]:
# Per night: hours that were fault/downtime AND outside weather = U(fault, downtime, planned, weather) - U(weather).
# Then split that into planned and unplanned. Planned takes precedence where the two overlap, so the parts add up:
#   planned   = U(planned, weather) - U(weather)
#   unplanned = not available - planned
not_available, planned = [], []
for d, w in nights.iterrows():
    per = periods_by_night[d]
    u_all = union_hours(
        per, NOT_AVAILABLE_STATES + WEATHER_STATES, w.night_start, w.night_end
    )
    u_planned = union_hours(
        per, PLANNED_STATES + WEATHER_STATES, w.night_start, w.night_end
    )
    not_available.append(
        u_all - w.weather_hours
    )  # w.weather_hours is U(weather) from Selection
    planned.append(u_planned - w.weather_hours)
nights["not_available_hours"] = np.clip(
    not_available, 0, None
)  # clip only removes rounding noise
nights["planned_hours"] = np.clip(planned, 0, None)
nights["unplanned_hours"] = nights.not_available_hours - nights.planned_hours

# Available = selected - not available, and the fraction of the selected time that was available.
nights["available_hours"] = nights.selected_hours - nights.not_available_hours
nights["availability"] = np.where(
    nights.selected_hours
    > 0.05,  # a night with no selected time (all weather) has no availability
    nights.available_hours / nights.selected_hours,
    np.nan,
)

display(
    nights[
        [
            "night_hours",
            "weather_hours",
            "selected_hours",
            "unplanned_hours",
            "planned_hours",
            "available_hours",
            "availability",
        ]
    ]
    .round(2)
    .head(10)
)

In [ ]:
# THE COMPUTATION: pool all nights. Total available hours over total selected hours (so a long night counts more
# than a short one, and nights with no selected time contribute nothing).
available_hours_total = nights.available_hours.sum()
selected_hours_total = nights.selected_hours.sum()
not_available_hours_total = nights.not_available_hours.sum()
planned_hours_total = nights.planned_hours.sum()
unplanned_hours_total = nights.unplanned_hours.sum()
availability = available_hours_total / selected_hours_total

print(f"selected time:                {selected_hours_total:8.1f} h")
print(
    f"not available, unplanned:     {unplanned_hours_total:8.1f} h  (FAULT, unplanned DOWNTIME)"
)
print(f"not available, planned:       {planned_hours_total:8.1f} h  (planned DOWNTIME)")
print(f"available:                    {available_hours_total:8.1f} h")
print(f"availability:                 {availability:8.1%}")

# Planned and unplanned downtime in nights, for comparison with the yearly allocations (<=14 scheduled, <=21 unscheduled).
# Each night's downtime is divided by that night's own observable-night length (a fully lost night counts as exactly 1,
# whatever the season). The range is not a full year: not prorated.
planned_nights = (nights.planned_hours / nights.night_hours).sum()
unplanned_nights = (nights.unplanned_hours / nights.night_hours).sum()
print(
    f"\nplanned downtime:   {planned_nights:5.1f} nights-equivalent (allocation: 14 per year)"
)
print(
    f"unplanned downtime: {unplanned_nights:5.1f} nights-equivalent (allocation: 21 per year)"
)

### Checks on the computation

Two diagnostics, not part of the metric, to show how much the choices above matter:

- **Fault or downtime hidden by weather.** Weather is removed first, so any `FAULT` or `DOWNTIME` that overlaps `WEATHER` is not charged to the system. This is intended (weather is not the engineers' doing), but if the recorded weather state persists through a long downtime, a lot of downtime can disappear from the metric this way. The size of that hidden time is shown below so it is never a surprise.
  - **Overrides that set both `WEATHER` and `DOWNTIME`** (15 July to 31 August) put every one of those nights wholly into this hidden time. The overrides from 1 September on choose one or the other per night from the weather loss file, so they do not.
- **Time with no recorded state.** The first night of a range can begin before the first status message, leaving time in which the state is unknown to the EFD record. Such time counts as available in the formula above (it is not recorded as fault or downtime), so its size is shown.

In [ ]:
# Diagnostic 1: hours of FAULT/DOWNTIME that overlap WEATHER (removed by the selection, not charged to the system).
# |(F or D) and W| = U(F or D) + U(W) - U(F or D or W)
hidden = []
unrecorded = []
for d, w in nights.iterrows():
    per = periods_by_night[d]
    u_fd = union_hours(per, NOT_AVAILABLE_STATES, w.night_start, w.night_end)
    u_all_states = union_hours(
        per,
        AVAILABLE_STATES + NOT_AVAILABLE_STATES + WEATHER_STATES,
        w.night_start,
        w.night_end,
    )
    hidden.append(
        u_fd
        + w.weather_hours
        - union_hours(
            per, NOT_AVAILABLE_STATES + WEATHER_STATES, w.night_start, w.night_end
        )
    )
    unrecorded.append(
        w.night_hours - u_all_states
    )  # Diagnostic 2: night time covered by no state at all
nights["fault_downtime_in_weather_hours"] = np.clip(hidden, 0, None)
nights["no_state_hours"] = np.clip(unrecorded, 0, None)

hidden_total = nights.fault_downtime_in_weather_hours.sum()
n_hidden_nights = int((nights.fault_downtime_in_weather_hours > 0.05).sum())
print(
    f"Fault/downtime overlapping weather (not charged): {hidden_total:.1f} h on {n_hidden_nights} nights "
    f"(vs {not_available_hours_total:.1f} h charged)"
)
print(
    f"Night time with no recorded state: {nights.no_state_hours.sum():.1f} h "
    f"on {int((nights.no_state_hours > 0.05).sum())} nights (counted as available)"
)

## Results

The result is reported two ways: a **headline statistic** compared with the 90% target, and a **calendar** showing the availability of every night.

### What is reported, and what cannot be yet

The metric definition asks for four things each cycle. Three can be reported today, one only in part.

| Reported each cycle | Status |
|---|---|
| Availability against 90%, with the observable-night definition stated | **Reported below.** Observable night = Sun -12 to -12 deg (ROSP-22), minus weather. |
| Scheduled downtime days against the 14-day allocation | **Reported in part.** Planned downtime is known only for the nights in the state overrides file; elsewhere it cannot be told apart from unplanned. Partial resolution of [ROSP-28](https://rubinobs.atlassian.net/browse/ROSP-28). |
| Unplanned downtime days against the 21-day allocation | **Reported in part**, for the same reason: unplanned is what is left after the planned nights are removed. |
| Subsystem attribution of unplanned downtime | **Not possible yet.** It needs the cause breakdown from time-loss accounting, pending [ROSP-24](https://rubinobs.atlassian.net/browse/ROSP-24) and [ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11). |

Planned and unplanned time **both count as not available** in the availability number below, so a scheduled maintenance window counts against the availability exactly like an unplanned failure. The planned and unplanned hours are shown separately, and the true availability against unplanned loss alone would be higher. Downtime is quoted in *nights-equivalent* (each night's downtime hours divided by that night's own observable-night length, so a fully lost night counts as 1), for a date range that is not a full year, so it is not prorated against the yearly allocations.

In [ ]:
# HEADLINE RESULT: availability against the 90% target, shown as a highlighted box so it stands out from the other output.
from IPython.display import HTML, display

AVAILABILITY_TARGET = 0.90  # target quoted in the metric definition: 90% availability of observable nights
PLANNED_ALLOCATION_NIGHTS = 14  # yearly allocation of scheduled downtime
UNPLANNED_ALLOCATION_NIGHTS = 21  # yearly allocation of unscheduled downtime

meets = availability >= AVAILABILITY_TARGET
verdict = "MEETS" if meets else "DOES NOT MEET"
accent = (
    "#1a7f37" if meets else "#c62828"
)  # the verdict is also written out, so colour is not the only cue

hidden_note = (
    f"<li>{hidden_total:.1f} h of FAULT/DOWNTIME overlapped WEATHER and was <b>not charged</b> to the system (see Checks on the computation).</li>"
    if hidden_total > 0.05
    else ""
)

display(
    HTML(
        f"""
<div style="border: 3px solid {accent}; border-radius: 8px; padding: 14px 20px; margin: 10px 0; max-width: 900px;
            background: rgba(128, 128, 128, 0.08); font-family: sans-serif;">
  <div style="font-size: 1.05em; font-weight: bold; letter-spacing: 0.04em;">
    ROSP-9 SYSTEM AVAILABILITY &mdash; dayObs {nights.index.min()} to {nights.index.max()} ({len(nights)} nights)
  </div>
  <div style="font-size: 2.6em; font-weight: bold; color: {accent}; margin: 6px 0 0 0;">{availability:.1%}</div>
  <div style="font-size: 1.2em; font-weight: bold; color: {accent};">
    {verdict} the {AVAILABILITY_TARGET:.0%} target &nbsp;({(availability - AVAILABILITY_TARGET) * 100:+.1f} percentage points)
  </div>
  <table style="margin: 10px 0; border-collapse: collapse;">
    <tr><td style="padding: 2px 16px 2px 0;">Observable night definition</td>
        <td><b>Sun {TWILIGHT_ALT:g} to {TWILIGHT_ALT:g} deg, minus weather</b></td></tr>
    <tr><td style="padding: 2px 16px 2px 0;">Selected time</td>
        <td><b>{selected_hours_total:.1f} h</b> ({total_weather:.1f} h of weather removed from {total_night:.1f} h of night)</td></tr>
    <tr><td style="padding: 2px 16px 2px 0;">Not available, unplanned</td>
        <td><b>{unplanned_hours_total:.1f} h</b> = {unplanned_nights:.1f} nights-equivalent (allocation {UNPLANNED_ALLOCATION_NIGHTS} per year)</td></tr>
    <tr><td style="padding: 2px 16px 2px 0;">Not available, planned</td>
        <td><b>{planned_hours_total:.1f} h</b> = {planned_nights:.1f} nights-equivalent (allocation {PLANNED_ALLOCATION_NIGHTS} per year)</td></tr>
  </table>
  <div style="font-weight: bold;">Caveats</div>
  <ul style="margin: 2px 0 0 0;">
    <li>Planned downtime comes from the observatory state overrides (partial resolution of ROSP-28); before 15 July and after 25 October the EFD state is used as logged, and <b>planned and unplanned are not distinguished there</b>.</li>
    <li>The availability above counts planned and unplanned downtime both as not available. The nights-equivalent figures cover this date range only, not a full year, so they are not prorated against the yearly allocations.</li>
    <li>Subsystem attribution is pending ROSP-24 and ROSP-11.</li>
    {hidden_note}
  </ul>
</div>
"""
    )
)

### Night-by-night calendar

The headline number hides how the availability is distributed. The calendar shows it for every night: one row per week (Monday to Sunday), one cell per night, the **percentage of the selected time that was available** written in the cell.

The colour scale is centred on the **90% target**:

- **Redder** the further a night falls *below* 90%, and **greener** the further it rises *above* 90%. A night close to 90% is pale grey.
- Both arms get darker with distance from the target, so the scale still reads if red and green look alike to you; the number in every cell says the same thing.
- Grey `wx` cells are nights with **no selected time** at all because the whole night was lost to weather, so they have no availability to show.
- Blue `pd` cells are nights that are mostly **planned downtime**. They are shown separately instead of as a red 0%, so planned work is not read as a failure. They still count as not available in the headline number.

This is the same calendar layout as in the ROSP-22 observable-night notebook.

In [ ]:
# Helpers for the calendar plot (adapted from the ROSP-22 notebook; notebooks here must not import from each other).
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize

# Diverging colour scale centred on the 90% target (the neutral grey sits at exactly 90): reds below it, greens above it.
# Position = availability / 100 on a Normalize(0, 100) scale, so the red arm spans 0-90% and the green arm 90-100%.
# Both arms get darker away from the target, so lightness as well as hue tells how far a night is from 90%.
AVAILABILITY_CMAP = LinearSegmentedColormap.from_list(
    "availability_vs_target",
    [
        (0.00, "#7a1414"),  # 0%: dark red
        (0.45, "#d03b3b"),  # 45%: red
        (0.90, "#f0efec"),  # 90%: neutral grey, the target
        (0.95, "#0ca30c"),  # 95%: green
        (1.00, "#066306"),  # 100%: dark green
    ],
)


def calendar_grid(values):
    """Weeks x 7 array (Monday to Sunday) of a per-night value (Series indexed by dayObs), and the first Monday."""
    days = pd.to_datetime(values.index.astype(str), format="%Y%m%d")
    first_monday = days.min() - pd.Timedelta(days=days.min().weekday())
    nweeks = (days.max() - first_monday).days // 7 + 1
    cal = np.full((nweeks, 7), np.nan)
    for day, v in zip(days, values.values):
        cal[(day - first_monday).days // 7, day.weekday()] = v
    return cal, first_monday


def draw_calendar(
    ax,
    values,
    cmap,
    norm,
    title,
    label=lambda v: f"{v:.0f}",
    ytick_every=4,
    fontsize=6.5,
):
    """Draw a calendar of a per-night value on `ax`, with the value written in every cell."""
    cal, first_monday = calendar_grid(values)
    nweeks = cal.shape[0]
    im = ax.imshow(np.ma.masked_invalid(cal), cmap=cmap, norm=norm, aspect="auto")
    for (i, j), v in np.ndenumerate(cal):
        if not np.isnan(v):
            r, g, b, _ = cmap(
                norm(v)
            )  # white text on dark cells, dark text on light cells
            dark = 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.4
            ax.text(
                j,
                i,
                label(v),
                ha="center",
                va="center",
                fontsize=fontsize,
                color="white" if dark else "#0b0b0b",
            )
    ax.set_xticks(range(7))
    ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], fontsize=8)
    ax.xaxis.tick_top()
    ticks = range(0, nweeks, ytick_every)
    ax.set_yticks(ticks)
    ax.set_yticklabels(
        [(first_monday + pd.Timedelta(weeks=int(i))).strftime("%b %d") for i in ticks],
        fontsize=8,
    )
    ax.set_xticks(np.arange(-0.5, 7, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, nweeks, 1), minor=True)
    ax.grid(which="minor", color="white", lw=1.5)
    ax.tick_params(which="both", length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, loc="left", fontsize=10, pad=24)
    return im

In [ ]:
# Calendar of the availability of each night, in percent of the selected time.
# Nights lost entirely to weather have no availability; give them -1 so they draw in the "under" colour (grey).
# Nights that are mostly planned downtime (more than half of the selected time) are drawn separately in blue, labelled "pd".
planned_night = (nights.selected_hours > 0.05) & (
    nights.planned_hours > 0.5 * nights.selected_hours
)
cal_values = (nights.availability * 100).fillna(-1).where(~planned_night)

cal_cmap = AVAILABILITY_CMAP.copy()
cal_cmap.set_under(
    "#c3c2b7"
)  # whole night lost to weather (mid grey, distinct from the near-white 90% midpoint)
cal_cmap.set_bad(
    "white"
)  # not a night in the range (and planned-downtime nights, overdrawn below)

fig, ax = plt.subplots(figsize=(6, 14))
im = draw_calendar(
    ax,
    cal_values,
    cal_cmap,
    Normalize(0, 100),
    f"System availability per night (% of observable night minus weather)\n"
    f"overall {availability:.1%} vs target {AVAILABILITY_TARGET:.0%}; planned downtime shown separately (pd)",
    label=lambda v: "wx" if v < 0 else f"{v:.0f}",
    ytick_every=2,
)

# Planned-downtime nights: one flat colour and the label "pd", so they are not read as an availability value.
planned_grid, _ = calendar_grid(planned_night.astype(float).where(planned_night))
ax.imshow(
    np.ma.masked_invalid(planned_grid),
    cmap=ListedColormap(["#9ecae1"]),
    vmin=0,
    vmax=1,
    aspect="auto",
)
for (i, j), v in np.ndenumerate(planned_grid):
    if not np.isnan(v):
        ax.text(j, i, "pd", ha="center", va="center", fontsize=6.5, color="#0b0b0b")

cbar = fig.colorbar(
    im,
    ax=ax,
    orientation="horizontal",
    fraction=0.025,
    pad=0.04,
    aspect=30,
    extend="min",
)
cbar.set_ticks([0, 25, 50, 75, AVAILABILITY_TARGET * 100, 100])
cbar.set_label(
    "Availability (%): red below the 90% target, green above\n"
    "grey wx = whole night lost to weather; blue pd = planned downtime",
    fontsize=9,
)

## Configuration-change markers

This section records **changes that can move the availability series for reasons other than how well the system performed**, so that a step in the numbers is not mistaken for a trend. There are two kinds:

- **Method changes**: a different observable-night definition, a different set of available / not-available states, a different treatment of weather or of the state record. By the PTH rules, a change of definition is a **series break**, not a restatement. The current method is **v1.0 (draft)**.
- **System changes**: software releases, hardware changes or reclassification of states that change what the EFD states mean.

### Current method decisions (v1.0 draft)

| Decision | Setting in this notebook | Status |
|---|---|---|
| Observable night | Sun -12 to -12 deg (`TWILIGHT_ALT`), from ROSP-22 | Decided, held fixed |
| Weather | Removed from the denominator (`WEATHER_STATES`). Where weather is logged it is removed, even if downtime overlaps it. | Decided |
| `UNKNOWN` | Counts as available | Decided |
| Available / not available | `OPERATIONAL`, `IDLE`, `UNKNOWN` / `FAULT`, `DOWNTIME`, `PLANNED_DOWNTIME` (`AVAILABLE_STATES`, `NOT_AVAILABLE_STATES`) | Decided: planned downtime **counts against** availability, and is reported separately from unplanned |
| Downtime in days | Each night's downtime hours divided by that night's observable-night length (nights-equivalent), not prorated | Decided |
| **State from 15 July 2026 to 25 October 2026** | **Replaced by the shared overrides file (`observatory_state_overrides.csv`)** | Decided. Nights are all weather or all downtime on purpose, and the weather loss file is a projection. |

**The state overrides.** From 15 July to 25 October 2026 the EFD state is replaced by the shared overrides file:

- 15 July to 31 August: `DOWNTIME | WEATHER`. An extreme weather event took the observatory down, so this period is deliberately **not counted against availability**.
- 1 to 14 September: `WEATHER` or `DOWNTIME` per night from the ROSP-53 weather loss file.
- 15 September to 4 October: planned `DOWNTIME` (20 scheduled nights).
- 5 to 25 October: `WEATHER` or `DOWNTIME` per night from the weather loss file.

Because weather is removed first, nights marked `WEATHER` drop out of the selection and their downtime is not charged to the system. **Recording this as a method change is deliberate:** any later change to the overrides file is a change of method and a **series break**. Record the new setting here with its date, and do not restate the earlier results as though they had used it.

The dictionaries below hold the changes as `{dayObs: description}`. `METHOD_CHANGES` is filled; `CONFIG_CHANGES` (system changes) is **empty** until the stewards agree what to record.

In [ ]:
# Method changes and system changes that affect how the availability series should be read: {dayObs: description}.
# Add new entries (with their date) when a decision changes; a change of method is a series break.
METHOD_CHANGES = {
    state_overrides.day_obs.min(): (
        "From this dayObs the EFD observatory state is replaced by the shared state overrides "
        "(observatory_state_overrides.csv). 20260715 to 20260831 is an extreme weather event (DOWNTIME | WEATHER) "
        "and is not counted against availability. Nights in WEATHER are removed from the selection, so their downtime "
        "is not charged to the system. Planned downtime is labelled separately (from 20260915). "
        "Any change to the overrides file is a series break."
    ),
}
CONFIG_CHANGES = (
    {}
)  # system / configuration changes (software, hardware, state reclassification): none recorded yet

for title, changes in [
    ("Method changes", METHOD_CHANGES),
    ("System changes", CONFIG_CHANGES),
]:
    in_range = {d: txt for d, txt in changes.items() if d <= nights.index.max()}
    print(f"{title}:")
    if in_range:
        for d, txt in sorted(in_range.items()):
            print(f"  {d}: {txt}")
    else:
        print("  none recorded for this range")